# CASM end-to-end verification notebook

Walks the full pipeline against a baseline observation. Designed for the user to inspect before any merge of the `refactor-tower` branch.

Stages (in order): CAsMan refresh → load + diagnostics → fringe-stop → SVD calibrate → BF weights + inspect + transit + deploy (DADA only) → imaging → layout-version trace.

Live deploy to `corr1`/`corr2` is **explicitly commented out**. Uncomment only when you intend to push weights live.

Run from the integration venv:

```bash
source /home/casm/software/dev/casm_venvs/casm_refactor_env/bin/activate
jupyter lab notebooks/end_to_end.ipynb
```

## 1. Setup

In [ ]:
import os
from pathlib import Path

%matplotlib inline

# Proxy env vars are required for run_sync_wiring's GitHub fetch.
os.environ.setdefault("HTTP_PROXY",  "http://10.70.0.10:8118")
os.environ.setdefault("HTTPS_PROXY", "http://10.70.0.10:8118")

# Optional: pin the canonical layout if you want a specific one.
# os.environ["CASM_LAYOUT_CSV"] = "/home/casm/software/dev/antenna_layouts/casm_antenna_layout_2026-05-07.csv"

# Imports across all five repos.
from casm_io.correlator import read_visibilities, AntennaMapping, load_format
from casm_io.constants import OVRO, C_LIGHT_M_PER_NS, resolve_layout_path

from casm_vis_analysis import (
    run_sync_wiring, run_build_layout,
    plot_autocorr, plot_waterfall, plot_fringe_diag, plot_phase_vs_freq,
    fringe_stop, FringeStoppedData,
    coherence_metric, auto_detect_sign,
    fit_delay, apply_delay,
    source_flux, RFIMask,
)
from casm_calibrator import (
    SVDCalibrator, SVDConfig, CalibrationResult, CalibrationWeightsWriter,
)
from bf_weights_generator import BFWeights, inspect_snap_weights
from casm_imaging import make_phased_sum_image

print('Layout will resolve to:', resolve_layout_path())
print('OVRO site:', OVRO)
print('C / 1 ns =', C_LIGHT_M_PER_NS, 'm')

## 2. CAsMan refresh (dry-run by default)

In [ ]:
# Pull a fresh CAsMan DB and show the wiring diff.
diff = run_sync_wiring(dry_run=True, force_pull=True)
print('added:',   len(diff['added']))
print('removed:', len(diff['removed']))
print('changed:', len(diff['changed']))

In [ ]:
# Build the consumer layout CSV (writes a dated file + updates the
# `current` symlink in $CASM_LAYOUT_DIR).
out = run_build_layout(dated=True, update_symlink=True, check_casman=True)
print(f"output: {out['output_csv']}  ({out['n_total']} rows, {out['n_active']} active)")
print(f"symlink updated: {out['symlink_updated']}")

## 3. Load + diagnostics

In [ ]:
# Pick the baseline observation here.
TIME_START = '2026-05-06 06:00:00'
TIME_END   = '2026-05-06 10:00:00'
TIME_TZ    = 'America/Los_Angeles'
DATA_ROOT  = '/mnt'
FORMAT     = '/home/casm/software/dev/casm_io/casm_io/correlator/configs/layout_64ant.json'

fmt = load_format(FORMAT)
ant = AntennaMapping.load()      # canonical resolver: $CASM_LAYOUT_CSV / current symlink
data = read_visibilities(
    time_start=TIME_START, time_end=TIME_END, time_tz=TIME_TZ,
    data_root=DATA_ROOT, fmt=fmt,
    freq_range_mhz=(60.0, 80.0),    # real on-disk channel skip via memmap
)
print(f'vis:       {data.vis.shape}')
print(f'freq_mhz:  {data.freq_mhz.shape}  [{data.freq_mhz[0]:.1f} -> {data.freq_mhz[-1]:.1f}]')
print(f'time_unix: {data.time_unix.shape}')
print(f'antenna mapping: {ant}')

In [ ]:
# Compose API: one read, many plots.
auto_figs = plot_autocorr(data, ant, scale='dB', ncols=3)
wf_figs   = plot_waterfall(data, ant, split_max=22)

## 4. Fringe-stop

In [ ]:
# For the fringe-stop wrapper, data must be ref+target-filtered.
# The simplest way is to re-read with explicit ref/targets, OR use
# the runner mirror which handles it for you. Demonstrating the runner
# here so the cell runs end-to-end without manual baseline indexing.
from casm_vis_analysis import run_fringe_stop
fs = run_fringe_stop(
    format=FORMAT, layout=str(out['output_csv']),
    ref_ant=10, source='sun', sign=-1,
    time_start=TIME_START, time_end=TIME_END, time_tz=TIME_TZ,
    data_root=DATA_ROOT,
    freq_range_mhz=(60.0, 80.0),
    delay_model=['linear'], antenna_delays=True,
    show=True,
)
print('fs keys:', list(fs.keys()))

## 5. SVD calibrate

The compose-style `svd_calibrate(fs, ant)` is staged but the full dict-wrapper lands with the VisibilityMatrix dedup. For the verification run, use `SVDCalibrator` directly with a `VisibilityMatrix` you build yourself (legacy path).

In [ ]:
# Stub: legacy SVD path (replace with svd_calibrate(fs, ant) when the
# matrix-construction wrapper lands on this branch).
# config = SVDConfig(threshold=4.0, ref_ant_idx=0)
# calibrator = SVDCalibrator(config)
# vis_matrix = ... # build (n_chan, n_ant, n_ant) Hermitian from fs
# svd_result = calibrator.calibrate(vis_matrix.vis_avg)
# CalibrationWeightsWriter().write(
#     'cal_demo.npz', svd_result, freqs_mhz=fs['freq_mhz'],
#     ant_ids=..., ref_ant_id=10, source='sun',
# )
print('SVD step is staged; uncomment once the dict-wrapper lands.')

## 6. Beamforming weights + inspect + source-transit + deploy

In [ ]:
# Inspect a known weights HDF5 (replace with the path your generate step writes).
# E.g. point at a previously-built file to validate the inspect API:
WEIGHTS_H5 = '/home/casm/software/dev/bf_weights_generator/int8_weights_mock_snap1.h5'
if Path(WEIGHTS_H5).exists():
    info = inspect_snap_weights(WEIGHTS_H5)
    print('non-zero SNAP inputs:', info['n_nonzero_snap_inputs'], '/ 64')
    print('compute layout:',  info['compute_layout'])
    print('output layout:',   info['output_layout'])
else:
    print(f'No weights HDF5 at {WEIGHTS_H5}; skipping inspect.')

In [ ]:
# Source transit: which beams does each source pass through over the day?
# Note: plot_source_transit promotion is deferred (see plan); use the
# example script for now:
#
#   python /home/casm/software/dev/bf_weights_generator/examples/plot_source_transit.py \
#       /path/to/weights.h5 --sources Sun Cas-A Cyg-A \
#       --time-tz America/Los_Angeles --time-start 06:00 --time-end 15:00
print('plot_source_transit promotion deferred; use examples/plot_source_transit.py for now.')

In [ ]:
# Generate weights (full path requires a cal NPZ from step 5).
# from bf_weights_generator import generate_combined_weights
# bf = generate_combined_weights(...)  # cal NPZ + array config
# bf.save_dada('./snap_outputs')
print('generate_combined_weights end-to-end pending cal NPZ from step 5.')

In [ ]:
# ----- LIVE DEPLOY: DO NOT RUN unless you intend to push to corr1/corr2 -----
# (deploy_bf_weights is uncommitted on production main; use the script form
# below until that file lands on the refactor-tower branch.)
#
# from bf_weights_generator import deploy_bf_weights
# deploy_bf_weights(bf, output_dir='./snap_outputs')          # DADA only
# deploy_bf_weights(bf, upload=True, save_defaults=True,      # LIVE
#                   utc_start='2026-MM-DD-HH:MM:SS')
#
# Or via the shell:
#   python /home/casm/software/dev/bf_weights_generator/deploy_bf_weights.py weights.h5
print('Live deploy cell intentionally commented out.')

## 7. Imaging

In [ ]:
# CLI mirror: `casm-bf-image --source sun --start ... --end ...`
img = make_phased_sum_image(
    source='sun',
    time_start=TIME_START,
    time_end=TIME_END,
    output_dir='./images',
    ang_max_deg=15.0,
    npix=101,
    fs_sign=-1,
    fast_mode=True,
    compute_snr=True,
    verbose=True,
)
if img.get('snr_info') is not None:
    print('Measured SNR:', img['snr_info']['snr'])

## 8. Layout-version trace

In [ ]:
# Confirm every artifact references the same canonical CSV.
print('canonical layout (resolved):', resolve_layout_path())
print('build_layout output_csv:    ', out['output_csv'])
# When cal NPZ + weights HDF5 are produced, also print their layout_version stamps:
# import numpy as np
# cal = np.load('cal_demo.npz', allow_pickle=True)
# print('cal layout_version:', dict(cal['layout_version'].item()))